# Build Processed Real Video Dataset

语义 Stage：`real_video_vae_latent_probe` 过渡准备。这个 notebook 只负责把已登记 raw dataset 构建为正式 processed dataset，具体构建逻辑由 `paper_workflow.notebook_utils.processed_real_video_dataset_workflow` 委托仓库脚本执行。

### 00 Runtime Identity And User Config (`00_runtime_identity_and_user_config`)

作用：声明本次数据构建 workflow 的身份、Drive 路径和 processed dataset key。

参数说明：`RAW_DATASET_KEY` 选择 raw registry 中的数据源；`PROCESSED_DATASET_KEY` 是下游 notebook 读取的正式数据集身份；`LOCAL_WORKSPACE_ROOT` 是 Colab 本地临时工作区，通常保持默认。

In [ ]:
from pathlib import Path
import json

from paper_workflow.notebook_utils import processed_real_video_dataset_workflow as dataset_workflow

WORKFLOW_KEY = 'processed_dataset_build'
RAW_DATASET_KEY = 'DAVIS_2017'
DRIVE_ROOT = Path('/content/drive/MyDrive')
RAW_DATASET_DOWNLOAD_MANIFEST_PATH = DRIVE_ROOT / 'Datasets' / 'raw_dataset_download_manifest.json'
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001'
PROCESSED_DATASET_ROOT = DRIVE_ROOT / 'TSTW' / 'datasets' / 'processed' / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = PROCESSED_DATASET_ROOT / 'dataset_manifest.json'
PROCESSED_DATASET_SUMMARY = PROCESSED_DATASET_ROOT / 'processed_dataset_summary.json'
PROCESSED_DATASET_CHECKS = PROCESSED_DATASET_ROOT / 'checks' / 'processed_dataset_checks.json'
LOCAL_WORKSPACE_ROOT = Path('/content/TSTW_runtime/raw_workspaces') / PROCESSED_DATASET_KEY

### 01 Mount Google Drive (`01_mount_google_drive`)

作用：挂载 Google Drive，后续只从 Drive 中读取已登记 raw artifact，并把 processed dataset 写回受治理目录。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### 02 Validate Raw Dataset Registry (`02_validate_raw_dataset_registry`)

作用：检查 `raw_dataset_download_manifest.json` 是否存在，并确认本次选择的 raw dataset registry 可读取。

In [ ]:
if not RAW_DATASET_DOWNLOAD_MANIFEST_PATH.exists():
    raise FileNotFoundError(RAW_DATASET_DOWNLOAD_MANIFEST_PATH)
raw_dataset_registry = json.loads(RAW_DATASET_DOWNLOAD_MANIFEST_PATH.read_text(encoding='utf-8'))
print({'raw_dataset_download_manifest.json': str(RAW_DATASET_DOWNLOAD_MANIFEST_PATH), 'dataset_key': RAW_DATASET_KEY})

### 03 Select Raw Dataset Source (`03_select_raw_dataset_source`)

作用：选择 raw archive 来源。

参数说明：`RAW_ARCHIVE_PATH = None` 表示优先使用 raw manifest 中登记的路径；如果要手动指定 archive，填入 Drive 中已登记、可审计的 archive 路径。

In [ ]:
RAW_ARCHIVE_PATH = None
print({'selected_raw_dataset_key': RAW_DATASET_KEY, 'raw_archive_override': RAW_ARCHIVE_PATH})

### 04 Prepare Local Dataset Workspace (`04_prepare_local_dataset_workspace`)

作用：创建本次 Colab 会话使用的本地工作区和 Drive processed dataset 目录；本地目录只作本次运行缓存，不作为跨 notebook handoff。

In [ ]:
LOCAL_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
print({'local_workspace_root': str(LOCAL_WORKSPACE_ROOT), 'processed_dataset_root': str(PROCESSED_DATASET_ROOT)})

### 05 Extract Raw Dataset Archive (`05_extract_raw_dataset_archive`)

作用：保留 raw archive 解压职责边界说明。实际解压和转换由 wrapper 调用仓库数据构建脚本完成，notebook cell 不直接实现数据处理逻辑。

In [ ]:
print('raw archive extraction is delegated to paper_workflow.notebook_utils.processed_real_video_dataset_workflow')

### 06 Build Processed Video Clips (`06_build_processed_video_clips`)

作用：调用 `dataset_workflow.build_processed_dataset_handoff(...)` 构建 processed video clips，并返回可序列化 handoff。

参数说明：输入路径来自 raw manifest 或 `RAW_ARCHIVE_PATH`；输出目录由 `PROCESSED_DATASET_ROOT` 控制；`clean_workspace=False` 表示不主动清理本地工作区，正式清理策略应在 wrapper 或脚本中治理。

In [ ]:
processed_dataset_handoff = dataset_workflow.build_processed_dataset_handoff(
    raw_dataset_download_manifest_path=RAW_DATASET_DOWNLOAD_MANIFEST_PATH,
    raw_dataset_key=RAW_DATASET_KEY,
    raw_archive_path=RAW_ARCHIVE_PATH,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_workspace_root=LOCAL_WORKSPACE_ROOT,
    clean_workspace=False,
)

### 07 Write Processed Dataset Manifest (`07_write_processed_dataset_manifest`)

作用：验证构建脚本已经写出 `dataset_manifest.json`，该 manifest 是下游 notebook 的正式数据输入依据。

In [ ]:
if not PROCESSED_DATASET_MANIFEST.exists():
    raise FileNotFoundError(PROCESSED_DATASET_MANIFEST)
print({'PROCESSED_DATASET_MANIFEST': str(PROCESSED_DATASET_MANIFEST)})

### 08 Validate Processed Dataset (`08_validate_processed_dataset`)

作用：读取 `processed_dataset_checks.json` 并执行硬门禁；checks 未通过时立即停止，不允许把未验证数据传给下游 notebook。

In [ ]:
processed_dataset_checks = json.loads(PROCESSED_DATASET_CHECKS.read_text(encoding='utf-8'))
if not processed_dataset_checks.get('status'):
    raise RuntimeError(processed_dataset_checks)
print({'processed_dataset_checks.json': str(PROCESSED_DATASET_CHECKS), 'status': processed_dataset_checks['status']})

### 09 Register Processed Dataset (`09_register_processed_dataset`)

作用：打印 wrapper 返回的 dataset registry handoff，确认 processed dataset 已完成登记。

In [ ]:
print({'dataset_registry.json': processed_dataset_handoff['dataset_registry.json']})

### 10 Print Processed Dataset Handoff (`10_print_processed_dataset_handoff`)

作用：输出下游 notebook 需要的最小 handoff 信息，包括 `PROCESSED_DATASET_KEY`、manifest、summary 和 checks 路径。

In [ ]:
print({
    'PROCESSED_DATASET_KEY': PROCESSED_DATASET_KEY,
    'PROCESSED_DATASET_ROOT': str(PROCESSED_DATASET_ROOT),
    'PROCESSED_DATASET_MANIFEST': str(PROCESSED_DATASET_MANIFEST),
    'processed_dataset_summary.json': str(PROCESSED_DATASET_SUMMARY),
    'processed_dataset_checks.json': str(PROCESSED_DATASET_CHECKS),
})